In [ ]:
# ============================================================
# OVERFLOW AND RECOVERY MECHANISMS
# ============================================================
#
# This notebook demonstrates the overflow phenomenon and the
# three recovery mechanisms discussed in the theory:
#
#     1. Saturation
#     2. Zeroing
#     3. Two's-complement overflow (wrap-around)
#
# HOW TO USE THE NOTEBOOK
#
# 1. Use "Input x" to change the value applied to the nonlinear
#    overflow element.
#
# 2. Use "Overflow limit M" to define the normal operating range
#
#                   -M <= x <= M.
#
# 3. Select one of the three overflow recovery mechanisms.
#
# 4. Inside the normal operating range, all mechanisms behave as
#
#                       Q(x) = x.
#
# 5. Outside this range, the response becomes nonlinear.
#
# 6. The red point shows the current operating point (x,Q(x)).
#
# 7. The notebook also displays the overflow error
#
#                       e(x) = Q(x) - x.
#
# IMPORTANT
#
# Quantization is deliberately ignored here. This notebook studies
# overflow independently, exactly as described in the theory.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive_output
from IPython.display import display


# ------------------------------------------------------------
# Fixed plotting limits
# ------------------------------------------------------------

X_MIN = -6.0
X_MAX = 6.0
Y_MIN = -6.0
Y_MAX = 6.0


# ------------------------------------------------------------
# Overflow functions
# ------------------------------------------------------------

def saturation_scalar(x, M):
    if x > M:
        return M

    if x < -M:
        return -M

    return x


def zeroing_scalar(x, M):
    if abs(x) <= M:
        return x

    return 0.0


def wrap_scalar(x, M):
    if abs(x) <= M:
        return x

    period = 2.0 * M

    return ((x + M) % period) - M


def overflow_function(x, M, method):
    if method == 'Saturation':
        return saturation_scalar(x, M)

    if method == 'Zeroing':
        return zeroing_scalar(x, M)

    return wrap_scalar(x, M)


def overflow_array(x, M, method):
    if method == 'Saturation':
        return np.clip(x, -M, M)

    if method == 'Zeroing':
        return np.where(np.abs(x) <= M, x, 0.0)

    y = ((x + M) % (2.0 * M)) - M

    y = np.where(np.abs(x) <= M, x, y)

    return y


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.ov-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.ov-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
    width: 960px;
}

.ov-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.ov-title {
    font-size: 16px;
    font-weight: bold;
    margin-bottom: 6px;
    color: #243447;
}

.ov-info {
    font-size: 14px;
    line-height: 1.55;
}

.ov-label {
    display: inline-block;
    min-width: 185px;
    font-weight: bold;
}

.ov-normal {
    color: #176b34;
    font-weight: bold;
}

.ov-overflow {
    color: #b21f2d;
    font-weight: bold;
}

.ov-value {
    font-size: 17px;
    font-weight: bold;
}

.ov-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 5px;
}

</style>
""")


# ------------------------------------------------------------
# Title and description
# ------------------------------------------------------------

title_html = HTML("""
<div class="ov-root">
    <div style="font-family:monospace;font-size:22px;font-weight:bold;margin-bottom:8px;">
        Overflow and Recovery Mechanisms
    </div>
</div>
""")


description_html = HTML("""
<div class="ov-root">

    <div class="ov-description">

        Overflow occurs when a signal value exceeds the range that can be represented by the digital system.
        Inside the normal operating region, the overflow element behaves as the identity function <b>Q(x) = x</b>.<br>

        Outside this region, the selected recovery mechanism modifies the signal and introduces a
        <b>nonlinear input-output characteristic</b>.<br>

        Move the input through the overflow boundaries and compare saturation, zeroing and
        two's-complement wrap-around.

    </div>

</div>
""")


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='305px')

slider_style = {'description_width': '120px'}


x_slider = FloatSlider(
    value=0.8,
    min=-5.5,
    max=5.5,
    step=0.05,
    description='Input x:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


M_slider = FloatSlider(
    value=2.0,
    min=0.5,
    max=3.0,
    step=0.25,
    description='Overflow limit M:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.2f'
)


method_selector = RadioButtons(
    options=['Saturation', 'Zeroing', "Two's-complement overflow"],
    value='Saturation',
    description='Method:',
    style={'description_width': '80px'},
    layout=Layout(width='305px')
)


controls_box = VBox(
    [
        HTML("<div class='ov-title'>Controls</div>"),
        x_slider,
        M_slider,
        method_selector
    ],
    layout=Layout(
        width='325px',
        min_width='325px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible'
    )
)


summary_html = HTML()


summary_html.layout = Layout(
    width='620px',
    min_width='620px',
    overflow='visible'
)


top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        align_items='flex-start',
        justify_content='space-between',
        overflow='visible'
    )
)


# ------------------------------------------------------------
# Interactive plot
# ------------------------------------------------------------

def update_overflow(x, M, method):

    y = overflow_function(x, M, method)

    error = y - x

    overflow = abs(x) > M

    if overflow:
        status_text = "<span class='ov-overflow'>OVERFLOW</span>"
    else:
        status_text = "<span class='ov-normal'>NORMAL LINEAR REGION</span>"


    summary_html.value = f"""
    <div class="ov-box">

        <div class="ov-title">
            Current Operating Point
        </div>

        <div class="ov-info">

            <span class="ov-label">Input</span>
            x = <span class="ov-value">{x:.4f}</span>
            <br>

            <span class="ov-label">Overflow limit</span>
            M = {M:.4f}
            <br>

            <span class="ov-label">Operating condition</span>
            {status_text}
            <br>

            <span class="ov-label">Recovery mechanism</span>
            {method}
            <br>

            <span class="ov-label">Output</span>
            Q(x) = <span class="ov-value">{y:.4f}</span>
            <br>

            <span class="ov-label">Overflow error</span>
            e(x) = Q(x) - x = {error:.4f}

        </div>

        <div class="ov-note">
            When |x| ≤ M, Q(x) = x and the element behaves linearly.
            When |x| > M, the recovery mechanism modifies the signal
            and the input-output relation becomes nonlinear.
        </div>

    </div>
    """


    xx = np.linspace(X_MIN, X_MAX, 3000)

    yy = overflow_array(xx, M, method)


    fig, ax = plt.subplots(figsize=(10.5, 4.8))

    ax.plot(xx, xx, '--', linewidth=1.3, label='Ideal linear response  y = x')

    ax.plot(xx, yy, linewidth=2.2, label=f'{method}:  y = Q(x)')

    ax.axvline(-M, linestyle=':', linewidth=1.2)

    ax.axvline(M, linestyle=':', linewidth=1.2)

    ax.axhline(0, linewidth=0.8)

    ax.axvspan(-M, M, alpha=0.08, label='Normal operating region')

    ax.plot(x, y, 'o', markersize=9, label='Current operating point')

    ax.set_xlim(X_MIN, X_MAX)

    ax.set_ylim(Y_MIN, Y_MAX)

    ax.set_xlabel('Input x')

    ax.set_ylabel('Output Q(x)')

    ax.set_title('Overflow Input-Output Characteristic')

    ax.grid(True, alpha=0.25)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))

    plt.tight_layout()

    plt.show()

    plt.close(fig)


interactive_plot = interactive_output(
    update_overflow,
    {
        'x': x_slider,
        'M': M_slider,
        'method': method_selector
    }
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(description_html)

display(top_row)

display(interactive_plot)